<a href="https://colab.research.google.com/github/vt214-cyber/project2-tcga-p53-capstone-project/blob/main/03_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Notebook 03 — Extension

**Worth 25 marks. Choose ONE.**

---

> **Read this first.** The extension is not required to do well. A
> complete, careful core — Notebooks 01 and 02, properly written and
> properly interpreted — is first-class work and scores around 70. This
> notebook is how you move from 70 to 85.
>
> **One extension. Not two, not all three.** Doing one well beats doing
> three badly, and there are no marks for volume.

All three use the same expression file as Notebook 02, and **none of
them need your earlier answers to be correct** — a working
`compare_groups` is given to you below.

## Every option has the same basic deliverables

Whichever option you choose, you must provide:

1. the named function, following the contract given;
2. the supplied structural checks passing;
3. **at least two useful checks of your own** — choose properties that
   would catch a plausible mistake, not merely `result is not None`;
4. a short interpretation explaining what the extension changes about
   your Notebook 02 conclusion.

The supplied checks are only a starting point. Choosing what else needs
testing is part of the extension.

> ### ⚠️ Run the setup, then only your chosen option
>
> Run the setup cells at the top first. After that, work through **Option
> A, B or C** — not all three. The untouched options still contain
> `NotImplementedError`, so **Restart and run all is not appropriate for
> this notebook** until you remove or skip the two options you did not
> choose.
>
> If you edit a function, re-run its cell before running its checks.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 86 of 93** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [10]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


# GIVEN - a working compare_groups, so the extension does not depend on
# whether yours came out right in Notebook 02. Use it.
def compare_groups(expr, group_col, group_a, group_b):
    """Compare expression between two groups with a Mann-Whitney U test."""
    a = expr.loc[expr[group_col] == group_a, "log2_tpm"].dropna()
    b = expr.loc[expr[group_col] == group_b, "log2_tpm"].dropna()
    if a.empty or b.empty:
        raise ValueError(
            f"empty group: {group_a!r} has {len(a)} rows, "
            f"{group_b!r} has {len(b)}"
        )
    p_value = float(mannwhitneyu(a, b, alternative="two-sided").pvalue)
    return {
        "n_a": int(len(a)), "n_b": int(len(b)),
        "median_a": float(a.median()), "median_b": float(b.median()),
        "difference": float(a.median() - b.median()),
        "p_value": p_value, "significant": bool(p_value < 0.05),
    }


STUDENT_ID = "vt214"   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

expr = pd.read_csv(fetch(f"expression_{GENE}_{CANCER_TYPE}.csv"))
tumours = expr[(expr["cancer_type"] == CANCER_TYPE)
               & (expr["group"] == "tumour")].copy()
print(len(tumours), "tumours in", CANCER_TYPE)

vt214: your gene is AEN, your cancer type is LUSC
498 tumours in LUSC


---

# Option C — Does it hold in a different tissue?

**The most straightforward, and available to everybody.**

You tested one cancer type. Is what you found a general property of p53
biology, or something specific to that tissue?

The expression file contains tumours from **twelve** cancer types.
Repeat your TP53 comparison in each and see where your gene behaves the
same way and where it does not.

Two things to be careful about, both marked:

- **Group sizes vary wildly.** TP53 is mutated in most oesophageal
  tumours and very few prostate ones. A comparison with eight mutant
  tumours is not evidence. Set a minimum and say what it is.
- **Twelve tests means twelve chances of a false positive.** At
  p < 0.05, you expect roughly one significant result by luck alone. Say
  something sensible about that — you need not apply a formal
  correction, but you do have to show you know the problem exists.

## The contract

```python
compare_across_cancers(expr, min_group_size)
```

**Takes** `expr`, the tidy expression table, and the minimum number of
mutant **and** wild-type tumours required for a cancer type to be kept.
Reuse the given `compare_groups` function for each eligible cancer.

**Returns** a DataFrame with exactly these columns:

```text
cancer_type, n_mut, n_wt, difference, p_value, significant
```

There should be one row per eligible cancer type, sorted by `p_value`
smallest first, with the index reset. `difference` is mutant minus
wild-type expression.

In [11]:
    # Step 1: Keep tumour sample only
    all_tumours = expr[expr["group"] == "tumour"]

    # Step 2: Display the cancer types in the data
    print(all_tumours["cancer_type"].unique())

    # Step 3: Count the tumour samples in each cancer type
    print(all_tumours["cancer_type"].value_counts())

['BRCA' 'COAD' 'ESCA' 'GBM' 'KICH' 'LGG' 'LIHC' 'LUAD' 'LUSC' 'PAAD'
 'PRAD' 'STAD']
cancer_type
BRCA    1092
LUAD     513
LGG      509
LUSC     498
PRAD     495
STAD     414
LIHC     369
COAD     288
ESCA     181
PAAD     178
GBM      153
KICH      66
Name: count, dtype: int64


In [12]:
 # Step 4: Go through each cancer type
for cancer in all_tumours["cancer_type"].unique():
        # Step 5: Select tumour samples for this cancer type
        cancer_data = all_tumours[all_tumours["cancer_type"] == cancer]

        # Step 6: Display the cancer type
        print(cancer)

        # Step 7: Count TP53 mutant and wild type samples
        print(cancer_data["tp53_status"].value_counts())
        print()

BRCA
tp53_status
WT     524
MUT    261
Name: count, dtype: int64

COAD
tp53_status
MUT    163
WT     106
Name: count, dtype: int64

ESCA
tp53_status
MUT    154
WT      26
Name: count, dtype: int64

GBM
tp53_status
WT     98
MUT    51
Name: count, dtype: int64

KICH
tp53_status
WT     45
MUT    21
Name: count, dtype: int64

LGG
tp53_status
WT     262
MUT    242
Name: count, dtype: int64

LIHC
tp53_status
WT     249
MUT    107
Name: count, dtype: int64

LUAD
tp53_status
MUT    257
WT     250
Name: count, dtype: int64

LUSC
tp53_status
MUT    396
WT      78
Name: count, dtype: int64

PAAD
tp53_status
MUT    103
WT      67
Name: count, dtype: int64

PRAD
tp53_status
WT     434
MUT     58
Name: count, dtype: int64

STAD
tp53_status
WT     210
MUT    200
Name: count, dtype: int64



In [13]:
def compare_across_cancers(expr, min_group_size):
    # Step 1: Keep tumour sample only
    all_tumours = expr[expr["group"] == "tumour"]
    # Step 2: Create a list to store the results
    rows = []

    # Step 3: Go through each cancer type
    for cancer in all_tumours["cancer_type"].unique():
        # Step 4: Select tumour samples for this cancer type
        cancer_data = all_tumours[all_tumours["cancer_type"] == cancer]
        # Step 5: Select TP53 mutant samples
        mutant = cancer_data[cancer_data["tp53_status"] == "MUT"]
        # Step 6: Select TP53 wild type samples
        wild_type = cancer_data[cancer_data["tp53_status"] == "WT"]
        # Step 7: Check that both groups meet the minimum size
        if len(mutant) >= min_group_size and len(wild_type) >= min_group_size:
            # Step 8: Compare mutant and wild type expression
            result = compare_groups(cancer_data, "tp53_status", "MUT", "WT")

            # Step 9: Store the results for this cancer type
            rows.append({
                "cancer_type": cancer,
                "n_mut": len(mutant),
                "n_wt": len(wild_type),
                "difference": result["difference"],
                "p_value": result["p_value"],
                "significant": result["significant"]
            })

    # Step 10: Convert the results into a DataFrame
    results = pd.DataFrame(rows)

    # Step 11: Sort by p-value, smallest first
    results = results.sort_values("p_value")

    # Step 12: Reset the row index
    results = results.reset_index(drop=True)

    # Step 13: Return the results
    return results

In [14]:
# CHECK - structure and internal consistency
across = compare_across_cancers(expr, 20)
EXPECTED_ACROSS_COLUMNS = ["cancer_type", "n_mut", "n_wt",
                           "difference", "p_value", "significant"]

run_checks({
    "returns a DataFrame with the required columns":
        lambda: (isinstance(across, pd.DataFrame)
                 and list(across.columns) == EXPECTED_ACROSS_COLUMNS),
    "each cancer type appears once":
        lambda: across["cancer_type"].is_unique,
    "every retained group meets the minimum size":
        lambda: (across[["n_mut", "n_wt"]] >= 20).all().all(),
    "rows are sorted by p-value, smallest first":
        lambda: across["p_value"].is_monotonic_increasing,
    "the index was reset":
        lambda: list(across.index) == list(range(len(across))),
    "significant agrees with p < 0.05 in every row":
        lambda: (across["significant"] == (across["p_value"] < 0.05)).all(),
})

✅ returns a DataFrame with the required columns
✅ each cancer type appears once
✅ every retained group meets the minimum size
✅ rows are sorted by p-value, smallest first
✅ the index was reset
✅ significant agrees with p < 0.05 in every row

6 of 6 checks passed


In [15]:
# YOUR CHECKS - add at least two checks that would catch a real mistake.
# Then use the remaining cells you need for your Option C analysis.
# Check 1: Check that LUSC appears in the results
print("LUSC included:", "LUSC" in across["cancer_type"].values)

# Check 2: Check that the LUSC expression difference is correct

# Step 1: Select LUSC samples
lusc_tumours = expr[expr["cancer_type"] == "LUSC"]
# Step 2: Keep tumour samples only
lusc_tumours = lusc_tumours[lusc_tumours["group"] == "tumour"]

# Step 3: Select TP53 mutant samples
lusc_mutant = lusc_tumours[lusc_tumours["tp53_status"] == "MUT"]

# Step 4: Select TP53 wild type samples
lusc_wild_type = lusc_tumours[lusc_tumours["tp53_status"] == "WT"]

# Step 5: Calculate the median expression in each group
mutant_median = lusc_mutant["log2_tpm"].median()
wild_type_median = lusc_wild_type["log2_tpm"].median()

# Step 6: Calculate mutant minus wild type
expected_difference = mutant_median - wild_type_median

# Step 7: Display the expected and function differences
print("Expected difference:", expected_difference)
print("Function difference:", lusc_result["difference"].values[0])

LUSC included: True
Expected difference: 0.04225000000000012
Function difference: 0.04225000000000012


In [16]:
# Display results for all cancer tpyes
display(across)

,cancer_type,n_mut,n_wt,difference,p_value,significant
0,LGG,242,262,-1.41915,7.146187e-48,True
1,PRAD,58,434,-0.53335,7.290796e-08,True
2,LIHC,107,249,-0.59160,1.270276e-07,True
3,LUAD,257,250,-0.28960,2.694344e-07,True
4,COAD,163,106,-0.40050,5.591067e-06,True
5,STAD,200,210,-0.32950,3.535465e-05,True
6,PAAD,103,67,-0.34540,1.047694e-04,True
7,GBM,51,98,-0.58750,3.247297e-04,True
8,KICH,21,45,-0.76180,1.270920e-02,True
9,BRCA,261,524,-0.10590,2.238133e-02,True


---

## Your write-up

Whichever option you chose: what you did, what you found, and — most
importantly — **what it changes about the answer you gave in Notebook
02.**

*Your extension write-up here.*

I compared AEN expression between TP53 mutant and TP53 wild type tumours across 12 different cancer types. I only included cancer types that had at least 20 samples in both groups.

The results showed that AEN expression was significantly lower in TP53 mutant tumours in 10 out of 12 cancer types. However, there was no significant difference in ESCA or LUSC. In LUSC, the difference was very small (0.04225) and the p-value was 0.8384.

This supports my original conclusion in Notebook 02, as there was no evidence of a difference in AEN expression between TP53 mutant and wild type tumours in LUSC. The results also show that this pattern is not the same across all cancer types, as most of the other cancers showed lower AEN expression in TP53 mutant tumours. However, this does not prove that TP53 mutations caused these changes in expression. Also, because I tested 12 cancer types, some significant results could have occurred by chance.